# Thống kê dữ liệu

Owner: Tiến. Mọi số liệu sinh từ `data/m1/papers_stats.csv` và `data/m1/references_stats.csv` lúc build.

Định nghĩa **bài thành công**: mọi version đều xử lý xong (`n_versions_ok == n_versions`), xem [Phương pháp](01-methodology.md).

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

DATA = Path("../data/m1")
FIG = Path("../figures")
FIG.mkdir(exist_ok=True)

df = pd.read_csv(DATA / "papers_stats.csv", dtype={"arxiv_id": str})
ok = df.n_versions_ok == df.n_versions

## Tổng quan

In [ ]:
summary = pd.Series(
    {
        "Số bài trong dải": len(df),
        "Số bài crawl thành công": int(ok.sum()),
        "Success rate (%)": round(100 * ok.mean(), 2),
        "Size TB trước xoá hình (MB)": round(df.size_before_bytes[ok].mean() / 1e6, 2),
        "Size TB sau xoá hình (MB)": round(df.size_after_bytes[ok].mean() / 1e6, 3),
        "#ref TB mỗi bài": round(df.n_refs_total.mean(), 1),
        "Success rate ref metadata (%)": round(100 * df.n_refs_meta_ok.sum() / df.n_refs_total.sum(), 2),
        "Thời gian TB mỗi bài (s)": round(df.process_sec.mean(), 2),
    }
)
summary.to_frame("Giá trị")

## Kích thước trước và sau khi xoá hình

In [ ]:
fig, ax = plt.subplots(figsize=(6, 3.5))
ax.hist(
    [df.size_before_bytes[ok] / 1e6, df.size_after_bytes[ok] / 1e6],
    bins=50,
    log=True,
    label=["Trước", "Sau"],
)
ax.set_xlabel("MB / bài")
ax.set_ylabel("Số bài (log)")
ax.legend()
fig.tight_layout()
fig.savefig(FIG / "m1_size_hist.svg")  # bản vector để dùng lại cho poster

## Số version và số reference

In [ ]:
df.n_versions.value_counts().sort_index().rename_axis("Số version").to_frame("Số bài")

In [ ]:
fig, ax = plt.subplots(figsize=(6, 3.5))
ax.hist(df.n_refs_total, bins=40)
ax.set_xlabel("Số reference / bài")
ax.set_ylabel("Số bài")
fig.tight_layout()
fig.savefig(FIG / "m1_refs_hist.svg")

## Reference metadata theo nguồn

In [ ]:
refs = pd.read_csv(DATA / "references_stats.csv")
refs["hit_rate_%"] = (100 * refs.n_hit / (refs.n_hit + refs.n_miss)).round(2)
refs